# StreetForward 缓存系统集成测试与演示（教学版）

本 Notebook 用于对 StreetForward 资产系统做**端到端集成验证**（Phase C2：asset-only 主链、`missing_policy=error` 全链路断 runtime），覆盖以下能力：

1. 资产导出（scene / segment，可选；资产已存在时可跳过）
2. 资产目录结构与文件完整性检查
3. `StreetForwardAssetStore` 导入与查询（manifest / index / pose / pointcloud / dynamic / `image_table`）
4. `MultiSceneDatasetV3`：`get_segment_index` / 点云 / batch 组装（像素由 `image_table` 路径加载；若磁盘文件分辨率与表内 height/width 不一致会自动缩放到训练分辨率以对齐内参；不依赖整 scene `DrivingDataset` 加载）
5. **Runtime 调用计数**：monkeypatch 证明 `_ensure_scene_loaded` / `_load_view_from_image_ref` 在 error 模式下为 0
6. 当前阶段 **test refs 主链关闭**：`resolve_test_image_refs_deterministic` 返回空；helper `resolve_test_image_refs_deterministic_from_sidx` 仅作对照
7. （可选）scheduler 一步取 batch smoke
8. **一致性对照**：构建 runtime dataset + asset dataset，并分别构建 scheduler，比对 batch 全字段（含图像/点云）差值

---

## 你将学到什么

- 如何用统一配置导出缓存资产
- 如何验证资产是否可被训练路径消费（含 error 策略与无 runtime 证明）
- 如何定位常见失败（缺少配置、缺少 parquet 引擎、asset 不匹配等）

> 说明：本 Notebook 假定你在 `conda drivestudio-new` 环境中运行，并且 `PYTHONPATH` 指向本仓库根目录（见第 0 节代码单元）。

In [1]:
# ====== 0) 环境准备 ======
# 建议在 notebook 内核中已经激活 conda 环境：drivestudio-new

import os
import json
import subprocess
from pathlib import Path
from pprint import pprint

import numpy as np


def run(cmd: str, check: bool = True):
    """运行 shell 命令并打印输出。"""
    print(f"\n[RUN] {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")
    return completed


# 强制设置项目路径，避免 import 歧义
PROJECT_ROOT = Path('/root/drivestudio-coding')
os.environ['PYTHONPATH'] = str(PROJECT_ROOT)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('PYTHONPATH   =', os.environ['PYTHONPATH'])

PROJECT_ROOT = /root/drivestudio-coding
PYTHONPATH   = /root/drivestudio-coding


## 1) 配置与路径检查

这里我们使用项目内置的完整配置：

- `tools/streetforward_assets_data_snippet.yaml`

它已包含：

- 顶层 `data`
- 顶层 `dataset`
- `data.assets`（导出/导入资产根目录、missing_policy 等）

这一步先做 **fast-fail**，避免跑很久后才发现配置错误。

In [2]:
from omegaconf import OmegaConf

CONFIG_PATH = PROJECT_ROOT / 'tools/streetforward_assets_data_snippet.yaml'
assert CONFIG_PATH.exists(), f'Config not found: {CONFIG_PATH}'

cfg = OmegaConf.load(str(CONFIG_PATH))
assert OmegaConf.select(cfg, 'data') is not None, '配置缺少顶层 data'
assert OmegaConf.select(cfg, 'dataset') is not None, '配置缺少顶层 dataset'
assert OmegaConf.select(cfg, 'data.assets.root') is not None, '配置缺少 data.assets.root'

ASSET_ROOT = Path(OmegaConf.select(cfg, 'data.assets.root'))
DATASET_NAME = str(OmegaConf.select(cfg, 'data.dataset'))
SCENE_ID = 0
SEGMENT_ID = 0

print('CONFIG_PATH =', CONFIG_PATH)
print('ASSET_ROOT  =', ASSET_ROOT)
print('DATASET     =', DATASET_NAME)
print('SCENE/SEG   =', SCENE_ID, SEGMENT_ID)

CONFIG_PATH = /root/drivestudio-coding/tools/streetforward_assets_data_snippet.yaml
ASSET_ROOT  = /root/autodl-tmp/streetforward_assets
DATASET     = nuscenes
SCENE/SEG   = 0 0


## 2) 资产导出（Scene + Segment）

这一步会调用两个 CLI：

1. `build_streetforward_scene_assets.py`
2. `build_streetforward_segment_assets.py`

> 提示：首次导出很慢（会加载数据并生成点云等）。下一节代码默认 **`RUN_EXPORT = False`** 跳过 CLI，避免误跑长任务；需要导出时改为 `True` 或复制打印出的命令到终端执行。
>
> 若 scene/segment 资产已存在于 `data.assets.root`，可直接做目录检查与后续验证。

In [3]:
scene_cmd = (
    f"cd {PROJECT_ROOT} && "
    f"PYTHONPATH={PROJECT_ROOT} "
    f"python tools/build_streetforward_scene_assets.py "
    f"--config_file {CONFIG_PATH} --scene_id {SCENE_ID}"
)
seg_cmd = (
    f"cd {PROJECT_ROOT} && "
    f"PYTHONPATH={PROJECT_ROOT} "
    f"python tools/build_streetforward_segment_assets.py "
    f"--config_file {CONFIG_PATH} --scene_id {SCENE_ID} --segment_id {SEGMENT_ID}"
)

# 首次导出很慢；若 scene_pool/segment_pool 下已有对应资产，可保持 False 跳过
RUN_EXPORT = False
if RUN_EXPORT:
    run(scene_cmd, check=True)
    run(seg_cmd, check=True)
else:
    print("[SKIP] RUN_EXPORT=False，跳过 CLI 导出（资产已存在时推荐）。")
    print("若尚未导出，请将 RUN_EXPORT=True 或在本机终端手动运行下面两条命令：")
    print(" ", scene_cmd)
    print(" ", seg_cmd)

[SKIP] RUN_EXPORT=False，跳过 CLI 导出（资产已存在时推荐）。
若尚未导出，请将 RUN_EXPORT=True 或在本机终端手动运行下面两条命令：
  cd /root/drivestudio-coding && PYTHONPATH=/root/drivestudio-coding python tools/build_streetforward_scene_assets.py --config_file /root/drivestudio-coding/tools/streetforward_assets_data_snippet.yaml --scene_id 0
  cd /root/drivestudio-coding && PYTHONPATH=/root/drivestudio-coding python tools/build_streetforward_segment_assets.py --config_file /root/drivestudio-coding/tools/streetforward_assets_data_snippet.yaml --scene_id 0 --segment_id 0


## 3) 资产目录结构检查

这里快速查看：

- `scene_pool/`
- `segment_pool/`
- `registries/`

并确认目标资产目录包含：

- `READY`
- `manifest.json`
- `image_table.parquet`（scene）
- `segment_index.npz/segment_pose.npz/pointcloud_*.npz/dynamic_tracks.npz`（segment）

In [4]:
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

scene_pool = ASSET_ROOT / 'scene_pool'
segment_pool = ASSET_ROOT / 'segment_pool'
registries = ASSET_ROOT / 'registries'

print('scene_pool exists   =', scene_pool.exists())
print('segment_pool exists =', segment_pool.exists())
print('registries exists   =', registries.exists())

scene_candidates = sorted([p for p in scene_pool.glob(f'scene-{DATASET_NAME}-{SCENE_ID:06d}-*') if p.is_dir()])
seg_candidates = sorted([p for p in segment_pool.glob(f'seg-{DATASET_NAME}-{SCENE_ID:06d}-{SEGMENT_ID:06d}-*') if p.is_dir()])

assert len(scene_candidates) > 0, '未找到 scene 资产目录'
assert len(seg_candidates) > 0, '未找到 segment 资产目录'

latest_scene = max(scene_candidates, key=lambda p: p.stat().st_mtime)
latest_seg = max(seg_candidates, key=lambda p: p.stat().st_mtime)

print('\nLatest scene asset   =', latest_scene)
print('Latest segment asset =', latest_seg)

scene_required = ['READY', 'manifest.json', 'scene_index.npz', 'image_table.parquet']
seg_required = ['READY', 'manifest.json', 'segment_index.npz', 'segment_pose.npz', 'pointcloud_static.npz', 'pointcloud_dynamic.npz', 'dynamic_tracks.npz']

for name in scene_required:
    assert (latest_scene / name).exists(), f'scene asset 缺少文件: {name}'
for name in seg_required:
    assert (latest_seg / name).exists(), f'segment asset 缺少文件: {name}'

print('\n✅ 资产目录结构检查通过')

scene_pool exists   = True
segment_pool exists = True
registries exists   = True

Latest scene asset   = /root/autodl-tmp/streetforward_assets/scene_pool/scene-nuscenes-000000-7587f567
Latest segment asset = /root/autodl-tmp/streetforward_assets/segment_pool/seg-nuscenes-000000-000000-2ee2dafb

✅ 资产目录结构检查通过


## 4) AssetStore 导入能力验证（核心 API）

这一节直接演示 `StreetForwardAssetStore`：

- `get_scene_asset(...).load_image_meta(refs)`
- `verify_segment_asset(...).load_*`

并打印关键统计，帮助你确认资产内容与预期一致。

In [5]:
from datasets.streetforward_assets import StreetForwardAssetStore

store = StreetForwardAssetStore(str(ASSET_ROOT), missing_policy='error')

scene_handle = store.get_scene_asset(DATASET_NAME, SCENE_ID)
scene_manifest = scene_handle.load_manifest()
print('scene asset_id =', scene_manifest['asset_id'])

# 取几个 refs 查 image_table
sample_refs = [(0, 0), (0, 1)]
meta_rows = scene_handle.load_image_meta(sample_refs)
print('\nimage_table sample rows:')
for row in meta_rows:
    print({
        'frame_idx': row['frame_idx'],
        'cam_id': row['cam_id'],
        'img_idx': row['img_idx'],
        'height': row['height'],
        'width': row['width'],
    })

seg_handle = store.verify_segment_asset(DATASET_NAME, SCENE_ID, SEGMENT_ID)
seg_manifest = seg_handle.load_manifest()
print('\nsegment asset_id =', seg_manifest['asset_id'])

sidx = seg_handle.load_segment_index()
pose = seg_handle.load_segment_pose()
pcd = seg_handle.load_pointcloud()
tracks = seg_handle.load_dynamic_tracks()

print('\nSegmentIndex:')
print('  num_cams         =', sidx['num_cams'])
print('  train_frames     =', len(sidx['frame_indices']))
print('  test_frames      =', len(sidx['test_frame_indices']))
print('  train_image_refs =', sidx['train_image_refs'].shape)
print('  test_image_refs  =', sidx['test_image_refs'].shape)

print('\nPose:')
print('  segment_first_frame_idx =', pose['segment_first_frame_idx'])
print('  segment_pose_source     =', pose['segment_pose_source'])
print('  world_to_seg0 shape     =', tuple(pose['world_to_seg0'].shape))

print('\nPointCloud:')
print('  background shape =', np.asarray(pcd['background']).shape)
print('  dynamic instances=', len(pcd.get('dynamic', {})))

print('\nDynamic Tracks:')
print('  frame_indices shape =', tracks['frame_indices'].shape)
print('  instance_intids     =', tracks['instance_intids'].shape)
print('  instances_quats     =', tracks['instances_quats'].shape)
print('  instances_trans     =', tracks['instances_trans'].shape)
print('  instances_fv        =', tracks['instances_fv'].shape)

assert 'image_table_version' in scene_manifest
assert 'pointcloud_config_normalized' in seg_manifest
print('\n✅ AssetStore 导入能力验证通过')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
scene asset_id = scene-nuscenes-000000-7587f567

image_table sample rows:
{'frame_idx': 0, 'cam_id': 0, 'img_idx': 0, 'height': 300, 'width': 533}
{'frame_idx': 0, 'cam_id': 1, 'img_idx': 1, 'height': 300, 'width': 533}

segment asset_id = seg-nuscenes-000000-000000-2ee2dafb

SegmentIndex:
  num_cams         = 6
  train_frames     = 105
  test_frames      = 105
  train_image_refs = (630, 2)
  test_image_refs  = (630, 2)

Pose:
  segment_first_frame_idx = 0
  segment_pose_source     = camera
  world_to_seg0 shape     = (4, 4)

PointCloud:
  background shape = (1800000, 6)
  dynamic instances= 16

Dynamic Tracks:
  frame_indices shape = (105,)
  instance_intids     = (16,)
  instances_quats     = (105, 16, 4)
  instances_trans     = (105, 16, 3)
  instances_fv        = (105, 16)

✅ AssetStore 导入能力验证通过


## 5) Dataset 侧资产优先路径验证（MultiSceneDatasetV3，Phase C2）

目标：在 `data.assets.use_prebuilt_assets=true` 且 `missing_policy=error` 下，验证**主链不加载整 scene runtime**，视图像素由 scene 资产 `image_table` 路径直读。

检查点：

1. `get_segment_index()` 来自 segment 资产（无需 `_ensure_scene_loaded`）
2. `get_segment_batch_from_image_refs(include_test=False)` 返回含 `pointcloud`；**不应**含 `test` 键（当前阶段主链关闭 test）
3. `resolve_test_image_refs_deterministic()` 主链返回 `[]`；对照用 `resolve_test_image_refs_deterministic_from_sidx(sidx)` 展示确定性展开（不参与训练主链）
4. 若 segment 有动态内容，可出现 `dynamic_info`（由 `dynamic_tracks` 等资产驱动）

> **`initialize()` 对本节不是必须的**：`get_segment_index` / `get_segment_batch_from_image_refs` 不检查 `_initialized`。显式调用 `initialize()` 会走 `MultiSceneDataset.initialize()` → 训练队列填充 + **`_preload_scenes()`**，容易提前把整 scene 拉进内存，与「资产主链尽量不加载整场景」相反。若你需要 scheduler / 与训练脚本一致的全局预热，再调用或交给 scheduler 构造时自动 `initialize()`。

In [6]:
import torch
from tools.train_minimal_streetforward_stage4_3_v4_common import build_multi_scene_dataset_v3
from datasets.multi_scene_dataset_v3 import BatchRequestV3

# 复用完整训练配置；首次构建可能较慢（取决于 data 配置是否触发 scene 预加载）
device = torch.device("cpu")
dataset = build_multi_scene_dataset_v3(cfg, device=device)
# 刻意不调用 initialize()：image-ref batch 主链不依赖 _initialized；避免 _preload_scenes() 提前加载整 scene。
# 需要 scheduler / 全量预热时再 dataset.initialize()（或 scheduler 构造时会自动 init）。

assert bool(getattr(dataset, "use_prebuilt_assets", False)), "演示需要 data.assets.use_prebuilt_assets=true"
assert str(getattr(dataset, "asset_missing_policy", "")) == "error", "演示需要 data.assets.missing_policy=error"

sidx = dataset.get_segment_index(SCENE_ID, SEGMENT_ID)
print("SegmentIndex from dataset:")
print("  train frames =", len(sidx.frame_indices))
print("  test frames  =", len(sidx.test_frame_indices))
print("  num cams     =", sidx.num_cams)

source_ref = tuple(sidx.train_image_refs[0]) if sidx.train_image_refs else (sidx.frame_indices[0], 0)
target_refs = [source_ref]
if len(sidx.frame_indices) > 1:
    target_refs.append((int(sidx.frame_indices[1]), int(source_ref[1])))

main_test = dataset.resolve_test_image_refs_deterministic(SCENE_ID, SEGMENT_ID)
helper_test = dataset.resolve_test_image_refs_deterministic_from_sidx(sidx)
print("resolve_test_image_refs_deterministic (主链，当前阶段) count =", len(main_test))
print("resolve_test_image_refs_deterministic_from_sidx (helper 对照) count =", len(helper_test))

req = BatchRequestV3(
    scene_id=int(SCENE_ID),
    segment_id=int(SEGMENT_ID),
    source_image_ref=(int(source_ref[0]), int(source_ref[1])),
    target_image_refs=[(int(r[0]), int(r[1])) for r in target_refs],
    include_test=False,
    test_image_refs=None,
)

batch = dataset.get_segment_batch_from_image_refs(req, enforce_target0_equals_source=True)

print("\nBatch keys:", sorted(batch.keys()))
assert "pointcloud" in batch
assert "test" not in batch, "Phase C2 主链不应包含 test 分支"
print("pointcloud background shape =", np.asarray(batch["pointcloud"]["background"]).shape)

if "dynamic_info" in batch:
    print("dynamic_info frames =", len(batch["dynamic_info"]))
else:
    print("dynamic_info not present (若本 segment 无动态或 tracks 未命中，可为空)")

print("\n✅ Dataset 资产优先路径验证通过")

SegmentIndex from dataset:
  train frames = 105
  test frames  = 105
  num cams     = 6
resolve_test_image_refs_deterministic (主链，当前阶段) count = 0
resolve_test_image_refs_deterministic_from_sidx (helper 对照) count = 630


/root/drivestudio-coding/datasets/multi_scene_dataset_v3.py:391: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  mask = torch.as_tensor(arr, dtype=torch.float32)



Batch keys: ['aabb', 'dynamic_info', 'index_meta', 'keyframe_info', 'pointcloud', 'request_meta', 'scene_folder_name', 'scene_id', 'segment_first_frame_idx', 'segment_first_pose', 'segment_first_pose_source', 'segment_id', 'source', 'target']
pointcloud background shape = (1800000, 6)
dynamic_info frames = 2

✅ Dataset 资产优先路径验证通过


## 6) Runtime 调用计数证明（error 模式）

对以下入口做 monkeypatch 计数，验证在 `missing_policy=error` 且 scene/segment 资产齐全时：

- `_ensure_scene_loaded`：整 scene runtime 加载，**应为 0**
- `_load_view_from_image_ref`：像素经 `DrivingDataset.pixel_source` 的旧路径，**应为 0**（主链走 `_load_view_from_asset_paths`）

同时确认：`resolve_test_image_refs_deterministic(...)` 主链返回 `[]`（当前阶段不取 test refs）。

> 注意：须先运行 **第 5 节** 构建好 `dataset`。

In [7]:
from types import MethodType

# 计数仅覆盖本 cell 内 patch 之后的那次 batch 组装（不包含上文 dataset.initialize 等）
runtime_counts = {
    "ensure_scene_loaded": 0,
    "load_view_from_image_ref": 0,
}

orig_ensure_scene_loaded = dataset._ensure_scene_loaded
orig_load_view_from_image_ref = dataset._load_view_from_image_ref


def _count_ensure_scene_loaded(self, scene_id):
    runtime_counts['ensure_scene_loaded'] += 1
    return orig_ensure_scene_loaded(scene_id)


def _count_load_view_from_image_ref(self, scene_dataset, image_ref):
    runtime_counts['load_view_from_image_ref'] += 1
    return orig_load_view_from_image_ref(scene_dataset, image_ref)


# monkeypatch runtime 入口
dataset._ensure_scene_loaded = MethodType(_count_ensure_scene_loaded, dataset)
dataset._load_view_from_image_ref = MethodType(_count_load_view_from_image_ref, dataset)

# 固定一个最小请求，验证主链可组 batch 且不触发 runtime
sidx_demo = dataset.get_segment_index(SCENE_ID, SEGMENT_ID)
assert len(sidx_demo.train_image_refs) > 0, 'train_image_refs 为空，无法演示'
source_ref = tuple(sidx_demo.train_image_refs[0])
req_runtime_proof = BatchRequestV3(
    scene_id=SCENE_ID,
    segment_id=SEGMENT_ID,
    source_image_ref=source_ref,
    target_image_refs=[source_ref],
    include_test=True,          # 会被当前阶段策略自动关闭
    test_image_refs=None,
)

batch_runtime_proof = dataset.get_segment_batch_from_image_refs(req_runtime_proof)

print('runtime counts =', runtime_counts)
print('request_meta.test_image_refs =', batch_runtime_proof['request_meta'].get('test_image_refs'))
print('resolve_test_image_refs_deterministic =', dataset.resolve_test_image_refs_deterministic(SCENE_ID, SEGMENT_ID))

assert runtime_counts['ensure_scene_loaded'] == 0, 'error 模式下不应触发 _ensure_scene_loaded'
assert runtime_counts['load_view_from_image_ref'] == 0, 'error 模式下不应触发 runtime 视图读取'
assert dataset.resolve_test_image_refs_deterministic(SCENE_ID, SEGMENT_ID) == [], '当前阶段默认不取 test refs'

# 恢复原始函数，避免影响后续 cell
dataset._ensure_scene_loaded = orig_ensure_scene_loaded
dataset._load_view_from_image_ref = orig_load_view_from_image_ref

print('✅ runtime 调用计数验证通过（error 模式 runtime 调用为 0）')

runtime counts = {'ensure_scene_loaded': 0, 'load_view_from_image_ref': 0}
request_meta.test_image_refs = None
resolve_test_image_refs_deterministic = []
✅ runtime 调用计数验证通过（error 模式 runtime 调用为 0）


## 7) （可选）Scheduler 一步取 batch 验证

演示 scheduler 仍通过原有 API 取 batch（内部已可走资产主链）。若只做资产与 dataloader 验证，可跳过。

> 须先运行 **第 5 节** 构建好 `dataset`。

In [8]:
from tools.train_minimal_streetforward_stage4_3_v4_common import build_train_scheduler_from_cfg

scheduler = build_train_scheduler_from_cfg(cfg, dataset)
out = scheduler.next_batch()

print('scheduler batch keys:', sorted(out.keys()))
assert 'pointcloud' in out
print('scheduler pointcloud background shape =', np.asarray(out['pointcloud']['background']).shape)
print('✅ Scheduler smoke passed')

Loading images:   0%|          | 0/196 [00:00<?, ?it/s]

Loading vehicle masks:  70%|███████   | 138/196 [00:00<00:00, 193.81it/s]


KeyboardInterrupt: 

## 8) Runtime vs Asset 一致性对照（dataset + scheduler）

目标：

1. 用同一份基础配置构建两套 `MultiSceneDatasetV3`：
   - runtime 版：`data.assets.enable=false`
   - asset 版：`data.assets.enable=true, use_prebuilt_assets=true, missing_policy=error`
2. **不调用** `dataset.initialize()`，直接用同一 `BatchRequestV3` 调 `get_segment_batch_from_image_refs` 对比全字段（递归比较，数值给出 max abs diff）——与资产主链「避免整 scene 预加载」一致。
3. 再分别构建 scheduler 并比较 `next_batch()`（注意：`TrainSchedulerV5` 构造时若 `dataset._initialized` 为假，**会内部调用** `dataset.initialize()`，因此 scheduler 段仍会触发训练队列与 `_preload_scenes()`，这是当前主训练栈的行为，不是 image-ref API 本身所需。）

说明：

- 为提高可重复性，scheduler 对比前会重设 `python/random`、`numpy`、`torch` 随机种子。
- 比较采用 `rtol=1e-4, atol=1e-5`；若不一致会直接抛错（fast-fail）。

In [ ]:
import random

import numpy as np
import torch
from omegaconf import OmegaConf

from datasets.multi_scene_dataset_v3 import BatchRequestV3
from tools.train_minimal_streetforward_stage4_3_v4_common import (
    build_multi_scene_dataset_v3,
    build_train_scheduler_from_cfg,
)


def clone_cfg_for_assets(base_cfg, *, enable_assets: bool):
    cfg_local = OmegaConf.create(OmegaConf.to_container(base_cfg, resolve=True))
    assert OmegaConf.select(cfg_local, "data.assets") is not None, "配置缺少 data.assets"
    cfg_local.data.assets.enable = bool(enable_assets)
    if enable_assets:
        cfg_local.data.assets.use_prebuilt_assets = True
        cfg_local.data.assets.missing_policy = "error"
    else:
        cfg_local.data.assets.use_prebuilt_assets = False
        cfg_local.data.assets.missing_policy = "ignore"
    return cfg_local


def _to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    if isinstance(x, np.ndarray):
        return x
    return None


def compare_nested(a, b, path="root", *, rtol=1e-4, atol=1e-5, diffs=None, numeric_stats=None):
    if diffs is None:
        diffs = []
    if numeric_stats is None:
        numeric_stats = []

    a_np = _to_numpy(a)
    b_np = _to_numpy(b)
    if a_np is not None or b_np is not None:
        if a_np is None or b_np is None:
            diffs.append((path, f"type mismatch: {type(a)} vs {type(b)}"))
            return diffs, numeric_stats
        if a_np.shape != b_np.shape:
            diffs.append((path, f"shape mismatch: {a_np.shape} vs {b_np.shape}"))
            return diffs, numeric_stats
        if a_np.size == 0 and b_np.size == 0:
            numeric_stats.append((path, 0.0))
            return diffs, numeric_stats

        a_num = a_np.astype(np.float64, copy=False)
        b_num = b_np.astype(np.float64, copy=False)
        abs_diff = np.abs(a_num - b_num)
        max_abs = float(abs_diff.max()) if abs_diff.size > 0 else 0.0
        numeric_stats.append((path, max_abs))
        if not np.allclose(a_num, b_num, rtol=rtol, atol=atol, equal_nan=True):
            diffs.append((path, f"numeric mismatch: max_abs_diff={max_abs:.6e}"))
        return diffs, numeric_stats

    if type(a) != type(b):
        diffs.append((path, f"type mismatch: {type(a)} vs {type(b)}"))
        return diffs, numeric_stats

    if isinstance(a, dict):
        ka = set(a.keys())
        kb = set(b.keys())
        if ka != kb:
            only_a = sorted(list(ka - kb))
            only_b = sorted(list(kb - ka))
            diffs.append((path, f"key mismatch only_a={only_a} only_b={only_b}"))
            return diffs, numeric_stats
        for k in sorted(ka):
            compare_nested(a[k], b[k], f"{path}.{k}", rtol=rtol, atol=atol, diffs=diffs, numeric_stats=numeric_stats)
        return diffs, numeric_stats

    if isinstance(a, (list, tuple)):
        if len(a) != len(b):
            diffs.append((path, f"length mismatch: {len(a)} vs {len(b)}"))
            return diffs, numeric_stats
        for i, (av, bv) in enumerate(zip(a, b)):
            compare_nested(av, bv, f"{path}[{i}]", rtol=rtol, atol=atol, diffs=diffs, numeric_stats=numeric_stats)
        return diffs, numeric_stats

    if isinstance(a, (str, int, bool, type(None))):
        if a != b:
            diffs.append((path, f"value mismatch: {a!r} vs {b!r}"))
        return diffs, numeric_stats

    if isinstance(a, float):
        max_abs = float(abs(a - b))
        numeric_stats.append((path, max_abs))
        if not np.isclose(a, b, rtol=rtol, atol=atol, equal_nan=True):
            diffs.append((path, f"float mismatch: abs_diff={max_abs:.6e}"))
        return diffs, numeric_stats

    if a != b:
        diffs.append((path, f"fallback mismatch: {a!r} vs {b!r}"))
    return diffs, numeric_stats


# ---- A) 构建两套 dataset（runtime vs asset）并比较同一请求 batch ----
cfg_runtime = clone_cfg_for_assets(cfg, enable_assets=False)
cfg_asset = clone_cfg_for_assets(cfg, enable_assets=True)

device = torch.device("cpu")
dataset_runtime = build_multi_scene_dataset_v3(cfg_runtime, device=device)
dataset_asset = build_multi_scene_dataset_v3(cfg_asset, device=device)

# 对比 A) 不 init：image-ref batch 不依赖 initialize，避免 _preload_scenes 提前加载整 scene。
print(
    "before batch compare: _initialized runtime / asset =",
    dataset_runtime._initialized,
    dataset_asset._initialized,
)

print("dataset_runtime.use_prebuilt_assets =", bool(getattr(dataset_runtime, "use_prebuilt_assets", False)))
print("dataset_asset.use_prebuilt_assets   =", bool(getattr(dataset_asset, "use_prebuilt_assets", False)))

sidx_asset = dataset_asset.get_segment_index(SCENE_ID, SEGMENT_ID)
source_ref = tuple(sidx_asset.train_image_refs[0]) if sidx_asset.train_image_refs else (sidx_asset.frame_indices[0], 0)
target_refs = [source_ref]
if len(sidx_asset.frame_indices) > 1:
    target_refs.append((int(sidx_asset.frame_indices[1]), int(source_ref[1])))

req_cmp = BatchRequestV3(
    scene_id=int(SCENE_ID),
    segment_id=int(SEGMENT_ID),
    source_image_ref=(int(source_ref[0]), int(source_ref[1])),
    target_image_refs=[(int(r[0]), int(r[1])) for r in target_refs],
    include_test=False,
    test_image_refs=None,
)

batch_runtime = dataset_runtime.get_segment_batch_from_image_refs(req_cmp, enforce_target0_equals_source=True)
batch_asset = dataset_asset.get_segment_batch_from_image_refs(req_cmp, enforce_target0_equals_source=True)

diffs_dataset, numeric_dataset = compare_nested(batch_runtime, batch_asset, path="dataset_batch")
max_abs_dataset = max((x[1] for x in numeric_dataset), default=0.0)

print("\n[Dataset Compare] numeric entries =", len(numeric_dataset))
print("[Dataset Compare] max abs diff   =", f"{max_abs_dataset:.6e}")
if diffs_dataset:
    print("[Dataset Compare] mismatches (first 20):")
    for p, msg in diffs_dataset[:20]:
        print(" -", p, "=>", msg)
assert len(diffs_dataset) == 0, f"dataset batch 不一致，mismatch_count={len(diffs_dataset)}"


# ---- B) 分别构建 scheduler 并比较 next_batch 输出 ----
scheduler_runtime = build_train_scheduler_from_cfg(cfg_runtime, dataset_runtime)
scheduler_asset = build_train_scheduler_from_cfg(cfg_asset, dataset_asset)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
out_runtime = scheduler_runtime.next_batch()

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
out_asset = scheduler_asset.next_batch()

diffs_sched, numeric_sched = compare_nested(out_runtime, out_asset, path="scheduler_batch")
max_abs_sched = max((x[1] for x in numeric_sched), default=0.0)

print("\n[Scheduler Compare] numeric entries =", len(numeric_sched))
print("[Scheduler Compare] max abs diff   =", f"{max_abs_sched:.6e}")
if diffs_sched:
    print("[Scheduler Compare] mismatches (first 20):")
    for p, msg in diffs_sched[:20]:
        print(" -", p, "=>", msg)
assert len(diffs_sched) == 0, f"scheduler batch 不一致，mismatch_count={len(diffs_sched)}"

print("\n✅ Runtime vs Asset 对照通过：dataset/scheduler batch 全字段一致（在设定容差内）")

before batch compare: _initialized runtime / asset = False False
dataset_runtime.use_prebuilt_assets = False
dataset_asset.use_prebuilt_assets   = True


ValueError: get_segment_index requires runtime scene loading, but missing_policy=error forbids runtime fallback.

## 9) 常见问题排查（FAQ）

### Q1: 报错 `Missing key dataset`
请确认传入的是**完整训练配置**（顶层含 `data` 和 `dataset`），推荐：

- `tools/streetforward_assets_data_snippet.yaml`

### Q2: 报错 parquet engine 缺失（`pyarrow` / `fastparquet`）
执行：

```bash
conda run -n drivestudio-new pip install pyarrow
```

### Q3: 报错 asset already exists / multiple assets
这是正常场景（历史导出存在）。Store 解析通常取**最新 mtime** 的资产目录。

### Q4: batch 中没有 `dynamic_info` 或没有 `test`
- **无 `test`**：Phase C2 当前阶段主链关闭 test refs，属预期。
- **无 `dynamic_info`**：若本 segment 无动态点云或无可用的 `dynamic_tracks` 对齐结果，可以为空；有非空 dynamic 点云时 error 模式要求 tracks 资产。

### Q5: 第 6 节 runtime 计数非 0
检查 `data.assets.missing_policy` 是否为 `error`、资产是否完整；若改为 `ignore`，会允许回退到 runtime。

---

## 结论

当本 Notebook 相关断言通过时，说明 StreetForward 资产系统从导出到 `MultiSceneDatasetV3` 的 **asset-only 主链**可用，且在 error 策略下可避免整 scene runtime 与旧视图加载路径。